# Notebook 01 - Non-Instruction Fine-Tuning (Stage 1)
## Healthcare FAQ Assistant · Practical Fine-Tuning Assignment 04

**Stage:** Domain Adaptive Continued Pretraining (DACP)  
**Goal:** Adapt a base LLM to healthcare vocabulary, writing style, and clinical knowledge  
**Method:** Causal Language Modelling on raw healthcare paragraphs (no instruction format)  
**Framework:** Unsloth + TRL SFTTrainer  

---
### What this notebook covers (rubric checklist)
- ✅ Step 2: Load and clean raw domain text  
- ✅ Step 3: Non-instruction fine-tuning with Unsloth  
- ✅ Load raw domain text  
- ✅ Clean and chunk text  
- ✅ Load base model using Unsloth  
- ✅ Apply LoRA / QLoRA  
- ✅ Train on raw text  
- ✅ Save adapter / model  
- ✅ Test model after non-instruction fine-tuning  

### Beyond rubric
- ✅ bf16/fp16 auto-detection  
- ✅ VRAM + training time benchmarking  
- ✅ Sequence packing (no padding waste)  
- ✅ Weights & Biases experiment tracking  
- ✅ save_pretrained_merged (float16 merge for next stage)  
- ✅ Text continuation quality test before and after training  


## Cell 1 — Install libraries

In [1]:
# ============================================================
# Install — pinned versions to prevent API breakage
# ============================================================
!pip install -q unsloth
!pip install -q "transformers==4.56.2"
!pip install -q --no-deps "trl==0.22.2"
!pip install -q datasets wandb rouge_score bert_score

# Install PDF, OCR, and Word dependencies
!pip install -q python-docx PyMuPDF
# !apt-get update -qq
# !apt-get install -y -qq tesseract-ocr
# !pip install -q ocrmypdf
!pip install -q marker-pdf

# Fix for Pillow/torchvision incompatibility
!pip install --upgrade --force-reinstall Pillow

print("Installation complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 115.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

## Cell 2 — Imports

In [1]:
# IMPORTANT: unsloth must be imported before transformers
import unsloth  # noqa: F401 — patches transformers in-place

import os
import re
import gc
import time
import json
import math
import unicodedata

from dataclasses import dataclass, field

# libraries for extraction
import fitz  # PyMuPDF
import tempfile
# import pytesseract
# from PIL import Image
# import io

# New for dealing with scanned documents not tesseract
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered
from docx import Document # python-docx

from typing import List, Dict, Any

import torch
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel, is_bfloat16_supported

import warnings
warnings.filterwarnings("ignore")

# ── GPU check ────────────────────────────────────────────────
assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime → Change runtime type → GPU (T4)"
)
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"   bf16 supported: {is_bfloat16_supported()}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ GPU: Tesla T4
   VRAM: 14.6 GB
   bf16 supported: False


## Cell 3 — Configuration
All hyperparameters in one place.

**Hyperparameter rationale:**
| Param | Value | Why |
|-------|-------|-----|
| `LORA_R` | 16 | Standard starting rank - expressive but memory-efficient |
| `LORA_ALPHA` | 32 | 2× rank = effective scaling factor of 2.0 |
| `LORA_DROPOUT` | 0 | Disabled for Stage 1 - short training, no overfitting risk |
| `STAGE1_LR` | 2e-4 | Standard LoRA LR - higher than full FT, adapter learns from zero |
| `BATCH_SIZE` | 1 | Maximum that fits on T4 at 512 tokens with QLoRA |
| `GRAD_ACCUM` | 8 | Effective batch = 1×8 = 8 examples per weight update |
| `MAX_STEPS` | 60 | Demo: 60 steps ≈ 4 min on T4. Remove for full training. |


In [3]:
# ============================================================
# GLOBAL CONFIGURATION
# ============================================================

@dataclass
class Config:
  # ── Model ────────────────────────────────────────────────────
  BASE_MODEL_NAME: str = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
  # Alternatives (all pre-quantized on Unsloth Hub):
  # "unsloth/tinyllama-bnb-4bit"          (1.1B — fastest)
  # "unsloth/Llama-3.2-1B-Instruct-bnb-4bit" (1B)
  # "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit" (0.5B)

  # ── Paths ────────────────────────────────────────────────────
  DATA_PATH: str           = "/content/data/non_instruction_data.txt"
  # DATA_PATH: str           = "/content/data/healthcare_generated_gpt.pdf"
  # DATA_PATH: str           = "/content/data/healthcare_generated_gpt_scanned.pdf"
  OUTPUT_ROOT: str         = "/content/healthcare_assistant"
  STAGE1_ADAPTER_DIR: str  = f"{OUTPUT_ROOT}/stage1_adapter"
  STAGE1_MERGED_DIR: str   = f"{OUTPUT_ROOT}/stage1_merged"

  # ── LoRA hyperparameters ─────────────────────────────────────
  LORA_R: int          = 16
  LORA_ALPHA: int      = 32
  LORA_DROPOUT: float  = 0
  LORA_TARGET_MODULES: list = field(default_factory=lambda: [
      "q_proj", "k_proj", "v_proj", "o_proj",
      "gate_proj", "up_proj", "down_proj",
  ])

  # ── Training hyperparameters ─────────────────────────────────
  MAX_SEQ_LENGTH: int  = 512
  BATCH_SIZE: int      = 1
  GRAD_ACCUM: int      = 8
  STAGE1_LR: float     = 2e-4
  WARMUP_STEPS: int    = 5
  LOGGING_STEPS: int   = 5
  MAX_STEPS: int       = 60
  NUM_EPOCHS: int      = 3
  SEED: int            = 42

  # ── W&B experiment tracking ──────────────────────────────────
  USE_WANDB: bool       = True
  WANDB_PROJECT: str    = "healthcare-assistant-ft"
  WANDB_RUN_NAME: str   = "stage1-non-instruction"

  # ── HuggingFace Hub (optional push) ─────────────────────────
  HF_USERNAME: str         = "ekblaise"
  HF_REPO_STAGE1_ADAPTER: str = f"{HF_USERNAME}/healthcare-qwen2.5-stage1-adapter"
  HF_REPO_STAGE1_MERGED: str  = f"{HF_USERNAME}/healthcare-qwen2.5-stage1-merged"


# =============================Initialize the config===============================
cfg = Config()

for path in [cfg.OUTPUT_ROOT, cfg.STAGE1_ADAPTER_DIR, cfg.STAGE1_MERGED_DIR]:
      os.makedirs(path, exist_ok=True)

print("✅ Configuration set.")
print(f"   Effective batch size: {cfg.BATCH_SIZE} × {cfg.GRAD_ACCUM} = {cfg.BATCH_SIZE * cfg.GRAD_ACCUM}")
print(f"   Model: {cfg.BASE_MODEL_NAME}")
print(f"   Adapter will be saved to: {cfg.STAGE1_ADAPTER_DIR}")

✅ Configuration set.
   Effective batch size: 1 × 8 = 8
   Model: unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit
   Adapter will be saved to: /content/healthcare_assistant/stage1_adapter


## Cell 4 — Weights & Biases (Tracking , monitoring)

In [4]:
# ============================================================
# Weights & Biases experiment tracking
# Gives you loss curves,VRAM graphs, and experiment comparison across runs.
# ============================================================

if cfg.USE_WANDB:
    import wandb
    wandb.login()   # Will prompt for API key
    print("✅ W&B logged in. Loss curves will appear at wandb.ai")
else:
    os.environ["WANDB_DISABLED"] = "true"
    print("ℹ️  W&B disabled. Set USE_WANDB=True in Cell 3 to enable.")


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ekblaise (ekblaise-krish-naik-academy) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ W&B logged in. Loss curves will appear at wandb.ai


## Cell 5 — Load and inspect raw domain text
The non-instruction dataset is plain text paragraphs about healthcare.
No instruction format, no Q&A structure — just raw domain prose.
The model learns vocabulary, writing style, and clinical facts.


In [6]:
# ============================================================
# Step 1: Load raw text file
# ============================================================
def load_and_extract_data(file_path: str) -> List[str]:
    # Scanity check to make sure file is present
    if not os.path.exists(file_path):
        raise FileNotFoundError(file_path)

    # split the text to get the extension
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".pdf":
        return extract_from_pdf(file_path)
    elif ext in [".docx", ".doc"]:
        return extract_from_docx(file_path)
    else:
        with open(file_path, "r", encoding="utf-8") as f:
            return [
                x.strip()
                for x in f.read().split("\n\n")
                if x.strip()
            ]

def extract_from_docx(path):
    doc = Document(path)
    return [
        p.text.strip()
        for p in doc.paragraphs
        if p.text.strip()
    ]

def extract_from_pdf(pdf_path):
    """
    Extract text using PyMuPDF when possible.
    If any page appears scanned, fall back to Marker for the entire PDF.
    """
    paragraphs = []
    scanned_detected = False

    with fitz.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf, start=1):
            text = page.get_text("text").strip()

            # Very little extractable text -> probably scanned
            if len(text) < 50:
                print(f"Page {page_num} appears scanned.")
                scanned_detected = True
                break

            paragraphs.extend(
                p.strip()
                for p in text.split("\n\n")
                if len(p.strip()) > 40
            )

    if not scanned_detected:
        print("Using PyMuPDF extraction.")
        return paragraphs

    print("Scanned pages detected. Using Marker OCR...")

    # Load models once
    converter = PdfConverter(
        artifact_dict=create_model_dict()
    )

    rendered = converter(pdf_path)
    markdown, _, _ = text_from_rendered(rendered)

    return [
        p.strip()
        for p in markdown.split("\n\n")
        if len(p.strip()) > 40
    ]


raw_paragraphs = load_and_extract_data(cfg.DATA_PATH)

all_paragraphs = [
    p.strip()
    for p in raw_paragraphs
    if len(p.strip()) > 60
]

print(f"✅ Raw text loaded.")
print(f"   File size: {os.path.getsize(cfg.DATA_PATH) / 1024:.1f} KB")
print(f"   Total paragraphs found: {len(all_paragraphs)}")
print(f"\n── First paragraph preview ──")
print(all_paragraphs[0][:400])

✅ Raw text loaded.
   File size: 55.3 KB
   Total paragraphs found: 110

── First paragraph preview ──
Patient-centred care is an approach that respects and responds to individual patient preferences, needs, and values, ensuring that patient values guide all clinical decisions. Its core dimensions include respect for patient values and expressed needs, emotional support and relief of fear and anxiety, information and communication in accessible language, coordination and integration of care across 


## Cell 6 — Text cleaning
Clean the raw text before tokenization.
Each cleaning step addresses a real artifact from PDF/text extraction.


In [8]:
# ============================================================
# Step 2: Clean and normalise text
# ============================================================

def clean_text(text: str) -> str:
    """
    Clean a paragraph of raw domain text.
    Steps and rationale:
      1. Unicode NFKC normalisation: converts ligatures, special chars
         to standard forms the tokenizer expects.
      2. Remove zero-width and BOM characters: invisible but confuse tokenizers.
      3. Heal hyphenated line breaks: 'pharma-\ncological' → 'pharmacological'
      4. Remove standalone page numbers: training noise with no semantic content.
      5. Normalise whitespace: multiple spaces/tabs → single space.
      6. Normalise paragraph breaks: 3+ newlines → 2 (single blank line).
      7. Merge within-paragraph newlines: single newline → space.
    """
    # 1. Unicode normalisation
    text = unicodedata.normalize("NFKC", text)
    # 2. Remove invisible characters
    text = text.replace("\u200b", "").replace("\ufeff", "").replace("\u00ad", "")
    # 3. Heal hyphenated line breaks
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    # 4. Remove standalone page numbers
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)
    # 5. Normalise horizontal whitespace
    text = re.sub(r"[ \t]+", " ", text)
    # 6. Normalise paragraph breaks
    text = re.sub(r"\n{3,}", "\n\n", text)
    # 7. Merge within-paragraph newlines to spaces
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    return text.strip()

# Apply cleaning to all paragraphs
cleaned_paragraphs = []
seen_keys = set()

for para in all_paragraphs:
    cleaned = clean_text(para)
    # Deduplicate: skip near-identical paragraphs
    dedup_key = cleaned[:70].lower().strip()
    if len(cleaned) >= 80 and dedup_key not in seen_keys:
        seen_keys.add(dedup_key)
        cleaned_paragraphs.append(cleaned)

print(f"✅ Cleaning complete.")
print(f"   Before: {len(all_paragraphs)} paragraphs")
print(f"   After:  {len(cleaned_paragraphs)} paragraphs (removed short/duplicate)")
print(f"\n── Sample cleaned paragraph ──")
print(cleaned_paragraphs[0])
print(f"\n   Character count: {len(cleaned_paragraphs[0])}")


✅ Cleaning complete.
   Before: 110 paragraphs
   After:  110 paragraphs (removed short/duplicate)

── Sample cleaned paragraph ──
Patient-centred care is an approach that respects and responds to individual patient preferences, needs, and values, ensuring that patient values guide all clinical decisions. Its core dimensions include respect for patient values and expressed needs, emotional support and relief of fear and anxiety, information and communication in accessible language, coordination and integration of care across settings, physical comfort and pain management, involvement of family and friends as appropriate, and access to care and continuity. Evidence consistently shows that patient-centred care improves patient satisfaction, treatment adherence, health outcomes, and reduces unnecessary investigations and readmissions.

   Character count: 712


## Cell 7 — Dataset statistics

In [9]:
# ============================================================
# Dataset quality statistics
# ============================================================
import statistics

char_counts = [len(p) for p in cleaned_paragraphs]
word_counts = [len(p.split()) for p in cleaned_paragraphs]

print("📊 Dataset Statistics")
print("=" * 50)
print(f"  Total paragraphs:      {len(cleaned_paragraphs):,}")
print(f"  Total characters:      {sum(char_counts):,}")
print(f"  Total words:           {sum(word_counts):,}")
print(f"")
print(f"  Paragraph length (chars):")
print(f"    Min:    {min(char_counts):,}")
print(f"    Max:    {max(char_counts):,}")
print(f"    Mean:   {statistics.mean(char_counts):.0f}")
print(f"    Median: {statistics.median(char_counts):.0f}")
print(f"")
print(f"  Paragraph length (words):")
print(f"    Min:    {min(word_counts)}")
print(f"    Max:    {max(word_counts)}")
print(f"    Mean:   {statistics.mean(word_counts):.0f}")
print(f"    Median: {statistics.median(word_counts):.0f}")
print(f"")
print(f"  Rubric requirement (≥50): {'✅ Met' if len(cleaned_paragraphs) >= 50 else '❌ Not met'}")
print(f"  2× target (≥100):         {'✅ Met' if len(cleaned_paragraphs) >= 100 else '❌ Not met'}")

# Save stats
stats = {
    "stage": "non_instruction",
    "paragraphs": len(cleaned_paragraphs),
    "total_chars": sum(char_counts),
    "total_words": sum(word_counts),
    "mean_chars": round(statistics.mean(char_counts), 1),
    "min_chars": min(char_counts),
    "max_chars": max(char_counts),
}
with open(f"{cfg.OUTPUT_ROOT}/stage1_data_stats.json", "w") as f:
    json.dump(stats, f, indent=2)
print(f"\n✅ Stats saved to {cfg.OUTPUT_ROOT}/stage1_data_stats.json")


📊 Dataset Statistics
  Total paragraphs:      110
  Total characters:      56,406
  Total words:           7,344

  Paragraph length (chars):
    Min:    348
    Max:    791
    Mean:   513
    Median: 478

  Paragraph length (words):
    Min:    48
    Max:    109
    Mean:   67
    Median: 61

  Rubric requirement (≥50): ✅ Met
  2× target (≥100):         ✅ Met

✅ Stats saved to /content/healthcare_assistant/stage1_data_stats.json


## Cell 8 — Build HuggingFace Dataset
Convert cleaned paragraphs into a HuggingFace Dataset object.
This integrates with SFTTrainer's tokenization and packing pipeline.


In [10]:
# ============================================================
# Build HuggingFace Dataset
# ============================================================
import random
random.seed(cfg.SEED)
random.shuffle(cleaned_paragraphs)

stage1_dataset = Dataset.from_list([{"text": p} for p in cleaned_paragraphs])

print(f"✅ Dataset created.")
print(stage1_dataset)
print(f"\nColumn names: {stage1_dataset.column_names}")
print(f"\nSample record:")
print(json.dumps(stage1_dataset[0], indent=2)[:400])


✅ Dataset created.
Dataset({
    features: ['text'],
    num_rows: 110
})

Column names: ['text']

Sample record:
{
  "text": "Chronic pain is defined as pain persisting beyond the normal expected healing time, typically more than 3 months. It affects approximately 28 million adults in the UK and has a significant biopsychosocial impact. It is now understood as a complex condition involving peripheral and central sensitisation, psychological factors including fear-avoidance beliefs and catastrophising, and so


## Cell 9 — Load base model with Unsloth (QLoRA)
FastLanguageModel.from_pretrained() in one call:
- Downloads Qwen2.5-1.5B-Instruct (pre-quantized 4-bit from Unsloth Hub)
- Applies NF4 quantization with double quantization
- Auto-detects bf16 (A100) or fp16 (T4)
- Returns model + tokenizer together

**Why QLoRA?** Stores weights in 4-bit (~700MB) but computes in float16.
With standard HuggingFace, the same model needs ~3GB in float16 + ~6GB for optimizer states.
QLoRA + paged AdamW fits the entire training run in a 15GB T4.


In [11]:
# ============================================================
# Step 3: Load base model using Unsloth (QLoRA)
# ============================================================
gc.collect()
torch.cuda.empty_cache()

print(f"Loading: {cfg.BASE_MODEL_NAME}")
print("This may take 1-2 minutes on first run (downloading weights)...\n")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = cfg.BASE_MODEL_NAME,
    max_seq_length = cfg.MAX_SEQ_LENGTH,
    dtype          = None,        # Auto: bf16 on A100, fp16 on T4
    load_in_4bit   = True,        # QLoRA: NF4 4-bit quantization
)

# Fix: LLaMA/Qwen-style models have no dedicated pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # Right-padding for causal LM training

# Disable KV-cache during training (conflicts with gradient checkpointing)
model.config.use_cache = False

print("\n✅ Base model loaded.")
print(f"   Model dtype: {next(model.parameters()).dtype}")
print(f"   Pad token: {tokenizer.pad_token!r} (ID: {tokenizer.pad_token_id})")
print(f"   Vocab size: {len(tokenizer):,}")


Loading: unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit
This may take 1-2 minutes on first run (downloading weights)...

==((====))==  Unsloth 2026.7.1: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

✅ Base model loaded.
   Model dtype: torch.float16
   Pad token: '<|vision_pad|>' (ID: 151654)
   Vocab size: 151,665


## Cell 10 — Apply LoRA adapters
FastLanguageModel.get_peft_model() in one call replaces:
- prepare_model_for_kbit_training()
- LoraConfig(...)
- get_peft_model()

Key argument: `use_gradient_checkpointing="unsloth"` — Unsloth's custom
gradient checkpointing saves up to 70% VRAM vs standard PyTorch checkpointing,
enabling sequences up to 4× longer on the same GPU.


In [12]:
# ============================================================
# Step 4: Apply LoRA / QLoRA adapters
# ============================================================
model = FastLanguageModel.get_peft_model(
    model,
    r                      = cfg.LORA_R,
    lora_alpha             = cfg.LORA_ALPHA,
    target_modules         = cfg.LORA_TARGET_MODULES,
    lora_dropout           = cfg.LORA_DROPOUT,
    bias                   = "none",
    use_gradient_checkpointing = "unsloth",  # Unsloth's 70% VRAM saving checkpoint
    random_state           = cfg.SEED,
)

model.print_trainable_parameters()

print(f"\n✅ LoRA adapters applied.")
print(f"   Rank (r):           {cfg.LORA_R}")
print(f"   Alpha:              {cfg.LORA_ALPHA}")
print(f"   Scaling (α/r):      {cfg.LORA_ALPHA / cfg.LORA_R:.1f}×")
print(f"   Dropout:            {cfg.LORA_DROPOUT}")
print(f"   Target modules:     {cfg.LORA_TARGET_MODULES}")


Unsloth 2026.7.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820

✅ LoRA adapters applied.
   Rank (r):           16
   Alpha:              32
   Scaling (α/r):      2.0×
   Dropout:            0
   Target modules:     ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


## Cell 11 — Baseline text continuation (before training)
Test the BASE model on healthcare text continuations before any training.
This establishes what the model already knows and what it lacks.
We will compare with the same prompts after Stage 1 training.


In [13]:
# ============================================================
# Baseline test: text continuation BEFORE training
# ============================================================

# Switch to inference mode for testing
FastLanguageModel.for_inference(model)

BASELINE_PROMPTS = [
    "Metformin is the first-line pharmacological treatment for type 2 diabetes because",
    "The primary mechanism of action of SGLT-2 inhibitors involves",
    "Diabetic ketoacidosis is a life-threatening emergency characterised by",
    "Blood pressure control in patients with diabetic nephropathy is essential because",
    "The CURB-65 score is used in clinical practice to",
]

def generate_continuation(prompt: str, max_new_tokens: int = 80) -> str:
    """Generate a text continuation using the current model state."""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens    = max_new_tokens,
            do_sample         = True,
            temperature       = 0.7,
            top_p             = 0.9,
            repetition_penalty= 1.1,
            pad_token_id      = tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()

print("=" * 70)
print("BASE MODEL — Text Continuations (before Stage 1 training)")
print("=" * 70)
baseline_results = {}
for prompt in BASELINE_PROMPTS:
    continuation = generate_continuation(prompt)
    baseline_results[prompt] = continuation
    print(f"\nPROMPT: {prompt}")
    print(f"CONTINUATION: {continuation}")
    print("-" * 60)

# Save baseline results for comparison
with open(f"{cfg.OUTPUT_ROOT}/stage1_baseline_continuations.json", "w") as f:
    json.dump(baseline_results, f, indent=2, ensure_ascii=False)
print(f"\n✅ Baseline results saved to {cfg.OUTPUT_ROOT}/stage1_baseline_continuations.json")

# Restore training mode
FastLanguageModel.for_training(model)


BASE MODEL — Text Continuations (before Stage 1 training)

PROMPT: Metformin is the first-line pharmacological treatment for type 2 diabetes because
CONTINUATION: of its beneficial effects on blood glucose levels. This effect results from a number of mechanisms including enhanced insulin sensitivity and improved insulin secretion, but it has also been suggested that it may have other anti-inflammatory properties.
The objective of this study was to determine if metformin had any effects on inflammation in diabetic rats. The researchers used adult Wistar Albino Rats (n=56) aged between
------------------------------------------------------------

PROMPT: The primary mechanism of action of SGLT-2 inhibitors involves
CONTINUATION: the reduction of blood glucose levels by which of the following actions?
A. Reducing insulin production
B. Increasing insulin sensitivity in peripheral tissues
C. Promoting the reabsorption of glucose in the renal tubules
D. Enhancing gluconeogenesis
Answer:

C



PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151654)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

## Cell 12 — Training configuration (SFTConfig)
Key Stage 1 training decisions:
- `packing=True` — concatenates short paragraphs into 512-token blocks (no padding waste, 2-3× more efficient)
- `learning_rate=2e-4` — high LR appropriate for LoRA adapters learning from a random initialisation
- `fp16/bf16` — auto-detected: bf16 on Ampere GPUs (A100, RTX3090+), fp16 on T4
- `optim="adamw_8bit"` — 8-bit optimizer reduces optimizer state memory by ~4×
- `report_to="wandb"` — loss curves logged to W&B if enabled


In [14]:
# ============================================================
# Step 5: Training configuration
# ============================================================
report_to = "wandb" if cfg.USE_WANDB else "none"

if cfg.USE_WANDB:
    import wandb
    wandb.init(project=cfg.WANDB_PROJECT, name=cfg.WANDB_RUN_NAME, config={
        "stage": "non_instruction",
        "model": cfg.BASE_MODEL_NAME,
        "lora_r": cfg.LORA_R,
        "lora_alpha": cfg.LORA_ALPHA,
        "lr": cfg.STAGE1_LR,
        "max_seq_length": cfg.MAX_SEQ_LENGTH,
        "max_steps": cfg.MAX_STEPS,
        "batch_size": cfg.BATCH_SIZE,
        "grad_accum": cfg.GRAD_ACCUM,
    })

stage1_config = SFTConfig(
    # ── Output ──────────────────────────────────────────────
    output_dir              = f"{cfg.OUTPUT_ROOT}/stage1_logs",

    # ── Steps / epochs ──────────────────────────────────────
    max_steps               = cfg.MAX_STEPS,    # Remove and use num_train_epochs for real training
    # num_train_epochs      = NUM_EPOCHS,   # Uncomment for real full training

    # ── Batch size ───────────────────────────────────────────
    per_device_train_batch_size = cfg.BATCH_SIZE,
    gradient_accumulation_steps = cfg.GRAD_ACCUM,

    # ── Learning rate ────────────────────────────────────────
    learning_rate           = cfg.STAGE1_LR,
    warmup_steps            = cfg.WARMUP_STEPS,
    lr_scheduler_type       = "cosine",     # Cosine decay: gentler than linear for causal LM

    # ── Precision ────────────────────────────────────────────
    fp16                    = not is_bfloat16_supported(),
    bf16                    = is_bfloat16_supported(),

    # ── Optimizer ────────────────────────────────────────────
    optim                   = "adamw_8bit", # 8-bit: 4× less VRAM for optimizer states

    # ── Regularisation ───────────────────────────────────────
    weight_decay            = 0.01,

    # ── Dataset ──────────────────────────────────────────────
    dataset_text_field      = "text",
    max_length              = cfg.MAX_SEQ_LENGTH,
    packing                 = True,         # Concatenate paragraphs → no padding waste

    # ── Logging / saving ─────────────────────────────────────
    logging_steps           = cfg.LOGGING_STEPS,
    logging_first_step      = True,
    save_strategy           = "no",         # No checkpoints for demo (saves time/disk)

    # ── Misc ─────────────────────────────────────────────────
    report_to               = report_to,
    seed                    = cfg.SEED,
    remove_unused_columns   = False,
)

print("✅ SFTConfig created.")
print(f"   Effective batch size: {cfg.BATCH_SIZE} × {cfg.GRAD_ACCUM} = {cfg.BATCH_SIZE * cfg.GRAD_ACCUM}")
print(f"   Precision: {'bf16' if is_bfloat16_supported() else 'fp16'}")
print(f"   Packing: True (no padding waste)")
print(f"   LR scheduler: cosine")
print(f"   Report to: {report_to}")


wandb: Detected [google.genai, huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


✅ SFTConfig created.
   Effective batch size: 1 × 8 = 8
   Precision: fp16
   Packing: True (no padding waste)
   LR scheduler: cosine
   Report to: wandb


## Cell 13 — Build SFTTrainer

In [15]:
# ============================================================
# Build trainer
# ============================================================
stage1_trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,   # trl >= 0.9 uses processing_class (not tokenizer)
    train_dataset    = stage1_dataset,
    args             = stage1_config,
)

# Count trainable parameters
total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ SFTTrainer ready.")
print(f"   Total parameters:     {total_params:,}")
print(f"   Trainable parameters: {train_params:,} ({100*train_params/total_params:.2f}%)")
print(f"   Training examples:    {len(stage1_dataset):,}")
print(f"   Max steps:            {cfg.MAX_STEPS}")
print(f"   Estimated time (T4):  ~{cfg.MAX_STEPS * 5 // 60}m {cfg.MAX_STEPS * 5 % 60}s")


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/110 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=6):   0%|          | 0/110 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!
✅ SFTTrainer ready.
   Total parameters:     907,081,216
   Trainable parameters: 18,464,768 (2.04%)
   Training examples:    110
   Max steps:            60
   Estimated time (T4):  ~5m 0s


## Cell 14 — Train (with VRAM + time benchmark)
The benchmark pattern:
1. `empty_cache()` — clear residual cached VRAM from data loading
2. `reset_peak_memory_stats()` — reset peak counter to measure training-only VRAM
3. `synchronize()` — wait for GPU to finish any queued work before starting clock
4. `trainer.train()`
5. `synchronize()` — wait for GPU to finish before stopping clock
6. Read peak VRAM and elapsed time


In [16]:
# ============================================================
# Step 5: Train on raw text (with VRAM + time benchmark)
# ============================================================
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

print("🚀 Starting Stage 1 training...")
print(f"   Model: {cfg.BASE_MODEL_NAME}")
print(f"   Steps: {cfg.MAX_STEPS} | Effective batch: {cfg.BATCH_SIZE * cfg.GRAD_ACCUM}")
print()

start_time = time.time()
train_result = stage1_trainer.train()
torch.cuda.synchronize()

train_time_sec     = round(time.time() - start_time, 1)
peak_allocated_gb  = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
peak_reserved_gb   = round(torch.cuda.max_memory_reserved()  / 1024**3, 3)

print()
print("=" * 55)
print("✅ STAGE 1 TRAINING COMPLETE")
print("=" * 55)
print(f"   Training time:        {train_time_sec:.1f}s ({train_time_sec/60:.1f} min)")
print(f"   Peak VRAM allocated:  {peak_allocated_gb:.3f} GB")
print(f"   Peak VRAM reserved:   {peak_reserved_gb:.3f} GB")
print(f"   Final train loss:     {train_result.training_loss:.4f}")
print("=" * 55)

# Save training metrics
metrics = {
    "stage": "non_instruction",
    "train_time_sec": train_time_sec,
    "peak_vram_allocated_gb": peak_allocated_gb,
    "peak_vram_reserved_gb": peak_reserved_gb,
    "final_train_loss": round(train_result.training_loss, 4),
    "max_steps": cfg.MAX_STEPS,
    "effective_batch_size": cfg.BATCH_SIZE * cfg.GRAD_ACCUM,
    "lora_r": cfg.LORA_R,
    "lora_alpha": cfg.LORA_ALPHA,
    "learning_rate": cfg.STAGE1_LR,
}
with open(f"{cfg.OUTPUT_ROOT}/stage1_training_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\n   Metrics saved to {cfg.OUTPUT_ROOT}/stage1_training_metrics.json")


🚀 Starting Stage 1 training...
   Model: unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit
   Steps: 60 | Effective batch: 8



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 25 | Num Epochs = 15 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
1,2.151800
5,2.265700
10,1.810000
15,1.460200
20,1.136000
25,0.827000
30,0.664700
35,0.458000
40,0.295900
45,0.245500



✅ STAGE 1 TRAINING COMPLETE
   Training time:        192.6s (3.2 min)
   Peak VRAM allocated:  4.849 GB
   Peak VRAM reserved:   4.982 GB
   Final train loss:     0.8081

   Metrics saved to /content/healthcare_assistant/stage1_training_metrics.json


## Cell 15 — Save LoRA adapter + merged model
Two saves:
1. **Adapter only** (~20-50MB) — save_pretrained() saves just the LoRA weights
2. **Merged model** (float16, ~3GB) — save_pretrained_merged() bakes adapter into base weights

The merged model is the input to Stage 2 — we always merge between stages to avoid
stacking adapters which compounds numerical errors.


In [17]:
# ============================================================
# Step 6: Save adapter and merged model
# ============================================================

# ── Save 1: LoRA adapter only (~20-50MB) ─────────────────────
print("💾 Saving LoRA adapter...")
model.save_pretrained(cfg.STAGE1_ADAPTER_DIR)
tokenizer.save_pretrained(cfg.STAGE1_ADAPTER_DIR)
adapter_files = os.listdir(cfg.STAGE1_ADAPTER_DIR)
print(f"✅ Adapter saved to: {cfg.STAGE1_ADAPTER_DIR}")
print(f"   Files: {adapter_files}")

# ── Save 2: Merged float16 model (for Stage 2) ───────────────
print("\n💾 Merging adapter into base model and saving...")
print("   This merges W_new = W_original + B×A×(alpha/r)")
print("   Result: standalone model with no PEFT overhead")
print("   (May take 2-3 minutes...)\n")

model.save_pretrained_merged(
    cfg.STAGE1_MERGED_DIR,
    tokenizer,
    save_method = "merged_16bit",   # Float16 — compatible with all downstream tools
)
print(f"\n✅ Merged model saved to: {cfg.STAGE1_MERGED_DIR}")
merged_size_gb = sum(
    os.path.getsize(os.path.join(cfg.STAGE1_MERGED_DIR, f))
    for f in os.listdir(cfg.STAGE1_MERGED_DIR)
    if os.path.isfile(os.path.join(cfg.STAGE1_MERGED_DIR, f))
) / 1024**3
print(f"   Merged model size: {merged_size_gb:.2f} GB")


💾 Saving LoRA adapter...
✅ Adapter saved to: /content/healthcare_assistant/stage1_adapter
   Files: ['README.md', 'tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja', 'special_tokens_map.json', 'vocab.json', 'merges.txt', 'added_tokens.json', 'adapter_model.safetensors', 'adapter_config.json']

💾 Merging adapter into base model and saving...
   This merges W_new = W_original + B×A×(alpha/r)
   Result: standalone model with no PEFT overhead
   (May take 2-3 minutes...)



config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [01:17<00:00, 77.46s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:03<00:00, 63.76s/it]


Unsloth: Merge process complete. Saved to `/content/healthcare_assistant/stage1_merged`

✅ Merged model saved to: /content/healthcare_assistant/stage1_merged
   Merged model size: 2.89 GB


## Cell 16 — Push to HuggingFace Hub

In [23]:
from huggingface_hub import notebook_login
notebook_login()    # Enter your HF token when prompted

In [24]:
# ============================================================
# Push to HuggingFace Hub for persistence
# Run this to save your model so Colab session loss doesn't
# mean re-running Stage 1 from scratch.
# ============================================================
PUSH_TO_HUB = True

if PUSH_TO_HUB:
    # Push adapter (small, fast)
    model.push_to_hub(cfg.HF_REPO_STAGE1_ADAPTER, private=True)
    tokenizer.push_to_hub(cfg.HF_REPO_STAGE1_ADAPTER, private=True)
    print(f"✅ Adapter pushed to: https://huggingface.co/{cfg.HF_REPO_STAGE1_ADAPTER}")

    # Push merged model (what Stage 2 will actually load)
    model.push_to_hub_merged(
        cfg.HF_REPO_STAGE1_MERGED,
        tokenizer,
        save_method="merged_16bit",
        private=True,
    )
    print(f"✅ Merged model pushed to: https://huggingface.co/{cfg.HF_REPO_STAGE1_MERGED}")
else:
    print("ℹ️  Hub push skipped. Set PUSH_TO_HUB=True to enable.")
    print(f"   Would push adapter to: {cfg.HF_REPO_STAGE1_ADAPTER}")
    print(f"   Would push merged to: {cfg.HF_REPO_STAGE1_MERGED}")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  32%|###2      | 23.9MB / 73.9MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Saved model to https://huggingface.co/ekblaise/healthcare-qwen2.5-stage1-adapter


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...iwbxajy/model.safetensors:   1%|          | 24.0MB / 3.09GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [02:18<00:00, 138.80s/it]


Unsloth: Merge process complete. Saved to `/tmp/tmpgiwbxajy`
✅ Merged model pushed to: https://huggingface.co/ekblaise/healthcare-qwen2.5-stage1-merged


## Cell 17 — Test after Stage 1 training
Compare text continuations before and after Stage 1.
After domain adaptation, the model should:
- Use more precise clinical terminology
- Continue medical sentences more naturally
- Show better domain coherence

Note: This is text CONTINUATION (not Q&A). Stage 2 teaches instruction following.


In [ ]:
# ============================================================
# Step 7: Test model after Stage 1 training
# ============================================================

# Switch to inference mode
FastLanguageModel.for_inference(model)

print("=" * 70)
print("STAGE 1 MODEL — Text Continuations (after non-instruction training)")
print("=" * 70)

post_results = {}
for prompt in BASELINE_PROMPTS:
    continuation = generate_continuation(prompt, max_new_tokens=100)
    post_results[prompt] = continuation
    baseline = baseline_results.get(prompt, "N/A")

    print(f"\nPROMPT: {prompt}")
    print(f"BEFORE: {baseline[:200]}")
    print(f"AFTER:  {continuation[:200]}")
    print("-" * 60)

# Save post-training results
with open(f"{cfg.OUTPUT_ROOT}/stage1_post_training_continuations.json", "w") as f:
    json.dump(post_results, f, indent=2, ensure_ascii=False)
print(f"\n✅ Post-training results saved.")

# Restore training mode for any subsequent operations
FastLanguageModel.for_training(model)


STAGE 1 MODEL — Text Continuations (after non-instruction training)

PROMPT: Metformin is the first-line pharmacological treatment for type 2 diabetes because
BEFORE: of its beneficial effects on blood glucose levels. This effect results from a number of mechanisms including enhanced insulin sensitivity and improved insulin secretion, but it has also been suggested
AFTER:  of its dose-proportional blood glucose-lowering effect without causing hypoglycemia. It works primarily by inhibiting hepatic gluconeogenesis through activation of AMP-activated protein kinase (AMPK),
------------------------------------------------------------

PROMPT: The primary mechanism of action of SGLT-2 inhibitors involves
BEFORE: the reduction of blood glucose levels by which of the following actions?
A. Reducing insulin production
B. Increasing insulin sensitivity in peripheral tissues
C. Promoting the reabsorption of glucose
AFTER:  insulin sensitisation, reducing peripheral resistance and lowering blood g

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151654)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora